# AI-Assisted Customer Support Analysis and Retrieval from Twitter

## Data Exploration

This notebook explores the Customer Support on Twitter dataset.

The raw dataset file `twcs.csv` is not stored in GitHub because it is too large.  
Instead, it should be downloaded from Kaggle and placed locally in the `data/` folder.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

### Dataset Overview

Before starting preprocessing, we first examine the structure of the dataset.  
This helps us understand what kind of data we are working with and which columns are useful for our task.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

In [ ]:
df = pd.read_csv("../Data/twcs.csv")

print("Preview data: \n", df.head())
print("Data size: \n", df.shape)
print("Columns names: \n", df.columns)
print("Data types: \n", df.info())

Preview data: 
    tweet_id   author_id  inbound                      created_at  \
0         1  sprintcare    False  Tue Oct 31 22:10:47 +0000 2017   
1         2      115712     True  Tue Oct 31 22:11:45 +0000 2017   
2         3      115712     True  Tue Oct 31 22:08:27 +0000 2017   
3         4  sprintcare    False  Tue Oct 31 21:54:49 +0000 2017   
4         5      115712     True  Tue Oct 31 21:49:35 +0000 2017   

                                                text response_tweet_id  \
0  @115712 I understand. I would like to assist y...                 2   
1      @sprintcare and how do you propose we do that               NaN   
2  @sprintcare I have sent several private messag...                 1   
3  @115712 Please send us a Private Message so th...                 3   
4                                 @sprintcare I did.                 4   

   in_response_to_tweet_id  
0                      3.0  
1                      1.0  
2                      4.0  
3             

The dataset contains over 2.8 million tweets, which is quite large.  
This means we will need to reduce the dataset size later for faster processing.

## Filtering Customer Tweets

The dataset contains both customer messages and company replies.  
Since the goal is to analyze customer support requests, we only keep tweets written by customers.

This is done using the `inbound` column:
- True → customer tweet  
- False → company response  

In [10]:
#Getting Customer tweets
customer_df = df[df["inbound"] == True].copy()

print("Customer tweets size : ",customer_df.shape)

Customer tweets size :  (1537843, 7)


In [ ]:
#Inbound count
df["inbound"].value_counts()

inbound
True     1537843
False    1273931
Name: count, dtype: int64

## Data Cleaning

The dataset is still large after filtering customer tweets.  
To make processing more efficient, we perform basic cleaning and reduce the dataset size.

Steps:
- remove missing text
- remove duplicate tweets
- sample a smaller subset for faster experimentation

In [11]:
# Removing missing text
customer_df = customer_df.dropna(subset=["text"])

# Removing duplicates
customer_df["text"].duplicated().sum()
customer_df = customer_df.drop_duplicates(subset=["text"])

customer_df.shape

(1511776, 7)

In [13]:
#Reduce Dataset
customer_df = customer_df.sample(n=200000, random_state=42)

print("Random Selected data:",customer_df.shape)

Random Selected data: (200000, 7)


## Extracting Company Labels

The dataset does not contain a direct category label.  
To create a supervised classification task, we use company mentions in the tweet text.

In [14]:
#Extract mentions
def extract_mentions(text):
    return re.findall(r"@(\w+)", str(text).lower())

customer_df["mentions"] = customer_df["text"].apply(extract_mentions)

customer_df[["text", "mentions"]].head(10)

,text,mentions
313121,@AskPlayStation @201034 I need help too man,"[askplaystation, 201034]"
1165223,"@AskeBay \nHello,\nI cannot find a human being...",[askebay]
208341,A turtle runs faster than my phone.. Bugs 11 s...,[115858]
240158,Another great set of @Delta flights thank to #...,[delta]
2637025,Amazonで頼んでたDVD届いてたぁーヾ(●´∇｀●)ﾉ\n早速【美女と野獣】観る(≧◡≦...,[]
325354,@115821 Why can't I talk to Alexa in the Alexa...,[115821]
378209,@119625 hi. Haikyu season 2 is available. But ...,[119625]
2467069,@UPSHelp Hi. We had 4 packages due for deliver...,[upshelp]
2322629,@AppleSupport why is my iPhone automatically g...,[applesupport]
2638307,@British_Airways I am unable to check-in onlin...,[british_airways]


In [16]:
#Keep tweets with mentions
customer_df = customer_df[customer_df["mentions"].apply(len) > 0].copy()

print("size of mentioned tweets: ", customer_df.shape)

size of mentioned tweets:  (192533, 8)


In [17]:
#Create label
customer_df["company"] = customer_df["mentions"].apply(lambda x: x[0])

customer_df[["text", "company"]].head(10)

,text,company
313121,@AskPlayStation @201034 I need help too man,askplaystation
1165223,"@AskeBay \nHello,\nI cannot find a human being...",askebay
208341,A turtle runs faster than my phone.. Bugs 11 s...,115858
240158,Another great set of @Delta flights thank to #...,delta
325354,@115821 Why can't I talk to Alexa in the Alexa...,115821
378209,@119625 hi. Haikyu season 2 is available. But ...,119625
2467069,@UPSHelp Hi. We had 4 packages due for deliver...,upshelp
2322629,@AppleSupport why is my iPhone automatically g...,applesupport
2638307,@British_Airways I am unable to check-in onlin...,british_airways
2219661,@Uber_Support Since yesterday I haven't receiv...,uber_support


In [18]:
#Top companies
customer_df["company"].value_counts().head(15)

#customer_df.shape

company
amazonhelp         16230
applesupport       10964
americanair         6049
delta               5385
uber_support        5344
southwestair        4311
115858              4285
virgintrains        4257
tesco               3888
spotifycares        3823
british_airways     3791
gwrhelp             3194
xboxsupport         3111
askplaystation      2752
chipotletweets      2704
Name: count, dtype: int64

## Selecting Top Companies

Some companies appear very rarely in the dataset.  
To make the classification more balanced and efficient, we will keep only top 10 most frequent companies.

In [ ]:
# Top 10 most frequent companies
top_companies = customer_df["company"].value_counts().head(10).index

# Filter dataset to keep only these companies
customer_df = customer_df[customer_df["company"].isin(top_companies)].copy()

# Check dataset size
print("Top Companies Size: ",customer_df.shape)


Top Companies Size:  (64536, 9)


In [25]:
# Show how many samples each company has
customer_df["company"].value_counts()

company
amazonhelp      16230
applesupport    10964
americanair      6049
delta            5385
uber_support     5344
southwestair     4311
115858           4285
virgintrains     4257
tesco            3888
spotifycares     3823
Name: count, dtype: int64

## Cleaning Company Labels

After extracting company mentions, we noticed that one of the labels is numeric (115858). 
Since our classification should focus on company accounts, we remove labels that only contain numbers before saving the final working dataset.

In [26]:
# Remove labels that are only numbers, because they are not company handles
customer_df = customer_df[~customer_df["company"].astype(str).str.isnumeric()].copy()

# Check new company distribution
print("Dataset shape after removing numeric labels:", customer_df.shape)
print(customer_df["company"].value_counts())

Dataset shape after removing numeric labels: (60251, 9)
company
amazonhelp      16230
applesupport    10964
americanair      6049
delta            5385
uber_support     5344
southwestair     4311
virgintrains     4257
tesco            3888
spotifycares     3823
Name: count, dtype: int64


In [27]:
# Save the filtered top-company dataset
customer_df.to_csv("../Data/top_company_tweets.csv", index=False)

print("Top company dataset saved successfully.")

Top company dataset saved successfully.


In [28]:
# Load the saved file to confirm it works
test_df = pd.read_csv("../Data/top_company_tweets.csv")

print("Saved file shape:", test_df.shape)
test_df.head()

Saved file shape: (60251, 9)


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,mentions,company
0,277379,182258,True,Fri Nov 03 15:02:36 +0000 2017,Another great set of @Delta flights thank to #...,277378,NaN,['delta'],delta
1,2488664,710705,True,Wed Nov 15 16:03:32 +0000 2017,@AppleSupport why is my iPhone automatically g...,2488662,NaN,['applesupport'],applesupport
2,2383194,646935,True,Fri Oct 20 07:09:21 +0000 2017,@Uber_Support Since yesterday I haven't receiv...,2383193,NaN,"['uber_support', '199518']",uber_support
3,1610886,332320,True,Thu Oct 19 19:30:27 +0000 2017,"@AmazonHelp There is no email from 2 days, jus...",1610887,1610885.0,"['amazonhelp', '115850']",amazonhelp
4,2763258,772995,True,Tue Nov 21 07:57:56 +0000 2017,"@SouthwestAir thanks for responding, will call...",2763257,2763259.0,['southwestair'],southwestair


## Text Cleaning

The tweet text contains mentions, links, punctuation, and other characters that may not be useful for modeling.  
We create a cleaned version of each tweet to use in the classification and search models.

In [ ]:
# Function to clean tweet text
def clean_text(text):
    text = str(text).lower()  # convert text to lowercase
    text = re.sub(r"http\S+", "", text)  # remove URLs
    text = re.sub(r"@\w+", "", text)  # remove user/company mentions
    text = re.sub(r"[^a-z\s]", "", text)  # keep only letters and spaces
    text = re.sub(r"\s+", " ", text).strip()  # remove extra spaces
    return text

# Apply the clean_text function to the original tweet text
customer_df["clean_text"] = customer_df["text"].apply(clean_text)

# Show original text and cleaned text side by side
customer_df[["text", "clean_text", "company"]].head()

,text,clean_text,company
240158,Another great set of @Delta flights thank to #...,another great set of flights thank to tmobilew...,delta
2322629,@AppleSupport why is my iPhone automatically g...,why is my iphone automatically going on mute,applesupport
2219661,@Uber_Support Since yesterday I haven't receiv...,since yesterday i havent received any update a...,uber_support
1466285,"@AmazonHelp There is no email from 2 days, jus...",there is no email from days just asked to wait...,amazonhelp
2592039,"@SouthwestAir thanks for responding, will call...",thanks for responding will call as soon as i g...,southwestair


In [30]:
# Remove rows where the cleaned text became empty
customer_df = customer_df[customer_df["clean_text"].str.len() > 0].copy()

print("Final dataset shape after text cleaning:", customer_df.shape)

Final dataset shape after text cleaning: (59184, 10)


In [31]:
# Save only the columns needed for modeling and search
customer_df[["tweet_id", "text", "clean_text", "company"]].to_csv(
    "../Data/final_tweets.csv",
    index=False
)

print("Final cleaned dataset saved successfully.")

Final cleaned dataset saved successfully.


In [32]:
# Load final dataset
final_df = pd.read_csv("../Data/final_tweets.csv")

print("Final saved file shape:", final_df.shape)
final_df.head()

Final saved file shape: (59184, 4)


,tweet_id,text,clean_text,company
0,277379,Another great set of @Delta flights thank to #...,another great set of flights thank to tmobilew...,delta
1,2488664,@AppleSupport why is my iPhone automatically g...,why is my iphone automatically going on mute,applesupport
2,2383194,@Uber_Support Since yesterday I haven't receiv...,since yesterday i havent received any update a...,uber_support
3,1610886,"@AmazonHelp There is no email from 2 days, jus...",there is no email from days just asked to wait...,amazonhelp
4,2763258,"@SouthwestAir thanks for responding, will call...",thanks for responding will call as soon as i g...,southwestair
